In [1]:
import os
from pathlib import Path
from ures.files import filter_files
from perf_estimator.estimator import Estimator, TrainerEstimator
from perf_estimator.dataset import image_dataset
from experiments.snapshot import SnapshotAnalyser
from perf_estimator.profiler import ProfilerDataProcessing
from ures.string import format_memory

from perf_estimator.utilis import format_memory


In [2]:
# Setup Basic Global Variables
root_dir = Path("/Users/jiaboshi/Documents/101-Data/002-xMem-LLM")
# root_dir = Path("/home/glaswigian/Documents/200-ResearchData/100-researchData/002-xMem-LLM/")
pytorch_dir = root_dir / "001-PyTorch"
huggingface_dir = root_dir / "002-HuggingFace"

pytorch_dir_name_format = "recurrence-{}-SGD-{}-1"
huggingface_dir_name_format_xmem = "{}-{}-xMem"
huggingface_dir_name_format_llm = "{}-{}-LLM"
huggingface_dir_name_format_cuda = "{}-{}-CUDA"

In [3]:
model = "VGG16"
batch_size = "170"
max_gpu_memory = 8

In [4]:
def get_xmem_memory(profiler_file, batch, max_in_gb, huggingface_enabled=False):
    if huggingface_enabled:
        estimator = TrainerEstimator(
            dataloader=image_dataset(batch=int(batch)),
            profiler_file=profiler_file,
            max_gpu_memory_in_gb=max_in_gb
        )
    else:
        estimator = Estimator(
            dataloader=image_dataset(batch=int(batch)),
            profiler_file=profiler_file,
            max_gpu_memory_in_gb=max_in_gb
        )
    my_result, _ = estimator.estimate()
    return max(my_result._trace.max_segment_changes), estimator


def get_ground_value_from_snapshot(snapshot_file: str) -> dict:
    _snapshot = SnapshotAnalyser(snapshot_file)
    return _snapshot.gpu_and_segment_in_same_time_length()

import re

def extract_bytes(mem_list):
    """
    从每个内存条目中提取 Bytes 数值，返回排序后的数值列表。
    假设每个内存条目是一个字符串，格式中包含 "Bytes:<数字>"
    """
    bytes_values = []
    pattern = re.compile(r'Bytes:(\d+)')
    for item in mem_list:
        # 将 item 转换为字符串以防万一
        text = str(item)
        match = pattern.search(text)
        if match:
            bytes_values.append(int(match.group(1)))
    return sorted(bytes_values)

def compare_memory_bytes(list1, list2, debug=False):
    """
    比较两个列表中各模块的 forward_memory 和 backward_memory 中的 bytes 数值是否一致
    """
    # 构造成字典，以模块名称为 key
    dict1 = {entry['name']: entry for entry in list1}
    dict2 = {entry['name']: entry for entry in list2}
    forward_matched = True
    backwrd_matched = True

    # 得到所有模块名称
    all_names = set(dict1.keys()).union(dict2.keys())

    for name in all_names:
        entry1 = dict1.get(name)
        entry2 = dict2.get(name)

        if entry1 is None:
            print(f"模块 {name} 仅存在于第二个列表中。")
            continue
        if entry2 is None:
            print(f"模块 {name} 仅存在于第一个列表中。")
            continue

        # 分别提取 forward_memory 和 backward_memory 中的 bytes 数值
        forward_bytes1 = extract_bytes(entry1.get('forward_memory', []))
        forward_bytes2 = extract_bytes(entry2.get('forward_memory', []))
        backward_bytes1 = extract_bytes(entry1.get('backward_memory', []))
        backward_bytes2 = extract_bytes(entry2.get('backward_memory', []))

        if forward_bytes1 == forward_bytes2:
            if debug:
                print(f"{name}: forward_memory 的 bytes 匹配")
        else:
            forward_matched = False
            if debug:
                print(f"{name}: forward_memory 的 bytes 不匹配")
                print(f"  List1: {forward_bytes1}")
                print(f"  List2: {forward_bytes2}")

        if backward_bytes1 == backward_bytes2:
            if debug:
                print(f"{name}: backward_memory 的 bytes 匹配")
        else:
            backwrd_matched = False
            if debug:
                print(f"{name}: backward_memory 的 bytes 不匹配")
                print(f"  List1: {backward_bytes1}")
                print(f"  List2: {backward_bytes2}")

    return forward_matched, backwrd_matched


In [5]:
from typing import Union
def get_all_memory_information(model_name: str, batch: Union[str, int]) -> dict:
    batch = str(batch)
    # Get PyTorch Data
    torch_data_dir = pytorch_dir.joinpath(pytorch_dir_name_format.format(model_name, batch))
    all_torch_dirs = os.listdir(torch_data_dir)
    all_torch_dirs = [torch_data_dir.joinpath(d) for d in all_torch_dirs if str(d).startswith(".") is False]
    dirs_sorted = sorted(all_torch_dirs, key=lambda d: d.stat().st_ctime)
    torch_snapshot_file = filter_files(".pickle", dirs_sorted[0], fuzz=True)[-1]
    torch_profiler_file = filter_files(".pt.trace.json", dirs_sorted[0], fuzz=True)[-1]

    # Get HuggingFace Data
    huggingface_dir_xmem = huggingface_dir.joinpath(huggingface_dir_name_format_xmem.format(model_name, batch))
    huggingface_dir_cuda = huggingface_dir.joinpath(huggingface_dir_name_format_cuda.format(model_name, batch))
    huggingface_dir_llm = huggingface_dir.joinpath(huggingface_dir_name_format_llm.format(model_name, batch))
    huggingface_profiler_file_xmem = filter_files(".pt.trace.json", huggingface_dir_xmem, fuzz=True)[-1]
    huggingface_profiler_file_llm = filter_files(".pt.trace.json", huggingface_dir_llm, fuzz=True)[-1]
    huggingface_snapshot_file_xmem = filter_files(".pickle", huggingface_dir_cuda, fuzz=True)[-1]

    # Estimate memory
    huggingface_memory_llm, huggingface_estimator = get_xmem_memory(
        profiler_file=huggingface_profiler_file_llm,
        batch=batch_size,
        max_in_gb=max_gpu_memory,
        huggingface_enabled=True
    )
    huggingface_snapshot_memory_xmen = max(get_ground_value_from_snapshot(huggingface_snapshot_file_xmem)['seg'])
    huggingface_memory_diff = huggingface_memory_llm - huggingface_snapshot_memory_xmen

    paper_memory_result, paper_estimator = get_xmem_memory(
        profiler_file=torch_profiler_file,
        batch=batch_size,
        max_in_gb=max_gpu_memory
    )
    paper_snapshot_result = max(get_ground_value_from_snapshot(torch_snapshot_file)['seg'])
    paper_memory_diff = paper_memory_result - paper_snapshot_result

    forward_matched, backward_matched = compare_memory_bytes(
        paper_estimator.profiler.get_iteration(1).layer_summary(),
        huggingface_estimator.profiler.get_iteration(1).layer_summary()
    )

    return {
        "torch": {
            "est": paper_memory_result,
            "ground": paper_snapshot_result,
            "error": round((abs(paper_memory_result - paper_snapshot_result)/paper_snapshot_result)*100, 2),
            "diff": paper_memory_diff
        },
        "huggingface": {
            "est": huggingface_memory_llm,
            "ground": huggingface_snapshot_memory_xmen,
            "error": round((abs(huggingface_memory_llm - huggingface_snapshot_memory_xmen)/huggingface_snapshot_memory_xmen)*100, 2),
            "diff": huggingface_memory_diff
        },
        "compare": {
            "forward": forward_matched,
            "backward": backward_matched,
            "est_diff": huggingface_memory_llm - paper_memory_result,
        }
    }


In [6]:
models = ["ConvNeXtTiny", "ResNet50", "VGG16"]
batch = range(10, 570, 40)

In [7]:
result_list = []
for m in models:
    for b in batch:
        _r = get_all_memory_information(m, b)
        _r.update({
            "model": m,
            "batch": b
        })
        result_list.append(_r)

Duplicate layer name found: ReLU_1. Renaming to ReLU_1_08
Duplicate layer name found: ReLU_1. Renaming to ReLU_1_c3
Duplicate layer name found: ReLU_2. Renaming to ReLU_2_98
Duplicate layer name found: ReLU_2. Renaming to ReLU_2_a3
Duplicate layer name found: ReLU_3. Renaming to ReLU_3_b4
Duplicate layer name found: ReLU_3. Renaming to ReLU_3_80
Duplicate layer name found: ReLU_4. Renaming to ReLU_4_60
Duplicate layer name found: ReLU_4. Renaming to ReLU_4_6d
Duplicate layer name found: ReLU_5. Renaming to ReLU_5_7c
Duplicate layer name found: ReLU_5. Renaming to ReLU_5_63
Duplicate layer name found: ReLU_6. Renaming to ReLU_6_dd
Duplicate layer name found: ReLU_6. Renaming to ReLU_6_ff
Duplicate layer name found: ReLU_7. Renaming to ReLU_7_99
Duplicate layer name found: ReLU_7. Renaming to ReLU_7_48
Duplicate layer name found: ReLU_8. Renaming to ReLU_8_87
Duplicate layer name found: ReLU_8. Renaming to ReLU_8_66
Duplicate layer name found: ReLU_9. Renaming to ReLU_9_8e
Duplicate laye

模块 ReLU_12_ff 仅存在于第一个列表中。
模块 ReLU_10_26 仅存在于第二个列表中。
模块 ReLU_14_a1 仅存在于第二个列表中。
模块 ReLU_9_ba 仅存在于第一个列表中。
模块 ReLU_5_63 仅存在于第二个列表中。
模块 ReLU_16_d5 仅存在于第二个列表中。
模块 ReLU_10_d2 仅存在于第一个列表中。
模块 ReLU_8_66 仅存在于第二个列表中。
模块 ReLU_12_c3 仅存在于第一个列表中。
模块 ReLU_1_08 仅存在于第二个列表中。
模块 ReLU_14_2a 仅存在于第一个列表中。
模块 ReLU_9_7a 仅存在于第一个列表中。
模块 ReLU_10_04 仅存在于第一个列表中。
模块 ReLU_4_24 仅存在于第一个列表中。
模块 ReLU_16_90 仅存在于第二个列表中。
模块 ReLU_3_82 仅存在于第一个列表中。
模块 ReLU_1_6c 仅存在于第一个列表中。
模块 ReLU_12_11 仅存在于第二个列表中。
模块 ReLU_2_a5 仅存在于第一个列表中。
模块 ReLU_11_35 仅存在于第二个列表中。
模块 ReLU_4_6d 仅存在于第二个列表中。
模块 ReLU_1_63 仅存在于第一个列表中。
模块 ReLU_3_b4 仅存在于第二个列表中。
模块 ReLU_13_21 仅存在于第一个列表中。
模块 ReLU_3_47 仅存在于第一个列表中。
模块 ReLU_15_8f 仅存在于第二个列表中。
模块 ReLU_6_42 仅存在于第一个列表中。
模块 ReLU_14_9d 仅存在于第一个列表中。
模块 ReLU_9_9a 仅存在于第二个列表中。
模块 ReLU_8_96 仅存在于第一个列表中。
模块 ReLU_16_9c 仅存在于第一个列表中。
模块 ReLU_7_19 仅存在于第一个列表中。
模块 ReLU_9_8e 仅存在于第二个列表中。
模块 ReLU_15_dd 仅存在于第一个列表中。
模块 ReLU_11_ec 仅存在于第一个列表中。
模块 ReLU_5_d0 仅存在于第一个列表中。
模块 ReLU_6_dd 仅存在于第二个列表中。
模块 ReLU_7_75 仅存在于第一个列表中。
模块 ReLU_13_f4 仅存在于第二个列表中。
模块 ReLU

Duplicate layer name found: ReLU_1. Renaming to ReLU_1_c8
Duplicate layer name found: ReLU_1. Renaming to ReLU_1_1e
Duplicate layer name found: ReLU_2. Renaming to ReLU_2_9c
Duplicate layer name found: ReLU_2. Renaming to ReLU_2_76
Duplicate layer name found: ReLU_3. Renaming to ReLU_3_24
Duplicate layer name found: ReLU_3. Renaming to ReLU_3_43
Duplicate layer name found: ReLU_4. Renaming to ReLU_4_2d
Duplicate layer name found: ReLU_4. Renaming to ReLU_4_23
Duplicate layer name found: ReLU_5. Renaming to ReLU_5_ce
Duplicate layer name found: ReLU_5. Renaming to ReLU_5_87
Duplicate layer name found: ReLU_6. Renaming to ReLU_6_ef
Duplicate layer name found: ReLU_6. Renaming to ReLU_6_4c
Duplicate layer name found: ReLU_7. Renaming to ReLU_7_fa
Duplicate layer name found: ReLU_7. Renaming to ReLU_7_2c
Duplicate layer name found: ReLU_8. Renaming to ReLU_8_38
Duplicate layer name found: ReLU_8. Renaming to ReLU_8_2e
Duplicate layer name found: ReLU_9. Renaming to ReLU_9_dc
Duplicate laye

模块 ReLU_16_ae 仅存在于第二个列表中。
模块 ReLU_16_e4 仅存在于第一个列表中。
模块 ReLU_13_28 仅存在于第二个列表中。
模块 ReLU_9_bb 仅存在于第一个列表中。
模块 ReLU_9_44 仅存在于第一个列表中。
模块 ReLU_15_e8 仅存在于第一个列表中。
模块 ReLU_11_37 仅存在于第一个列表中。
模块 ReLU_13_45 仅存在于第二个列表中。
模块 ReLU_6_ef 仅存在于第二个列表中。
模块 ReLU_6_4c 仅存在于第二个列表中。
模块 ReLU_11_73 仅存在于第二个列表中。
模块 ReLU_1_51 仅存在于第一个列表中。
模块 ReLU_10_01 仅存在于第一个列表中。
模块 ReLU_7_71 仅存在于第一个列表中。
模块 ReLU_13_6d 仅存在于第一个列表中。
模块 ReLU_15_85 仅存在于第二个列表中。
模块 ReLU_10_04 仅存在于第一个列表中。
模块 ReLU_1_c8 仅存在于第二个列表中。
模块 ReLU_1_1e 仅存在于第二个列表中。
模块 ReLU_4_23 仅存在于第二个列表中。
模块 ReLU_8_2e 仅存在于第二个列表中。
模块 ReLU_3_43 仅存在于第二个列表中。
模块 ReLU_9_dc 仅存在于第二个列表中。
模块 ReLU_11_8f 仅存在于第一个列表中。
模块 ReLU_4_2d 仅存在于第二个列表中。
模块 ReLU_10_c9 仅存在于第二个列表中。
模块 ReLU_7_df 仅存在于第一个列表中。
模块 ReLU_1_9a 仅存在于第一个列表中。
模块 ReLU_2_52 仅存在于第一个列表中。
模块 ReLU_5_87 仅存在于第二个列表中。
模块 ReLU_5_46 仅存在于第一个列表中。
模块 ReLU_2_bd 仅存在于第一个列表中。
模块 ReLU_4_67 仅存在于第一个列表中。
模块 ReLU_5_ce 仅存在于第二个列表中。
模块 ReLU_11_7e 仅存在于第二个列表中。
模块 ReLU_13_85 仅存在于第一个列表中。
模块 ReLU_7_2c 仅存在于第二个列表中。
模块 ReLU_5_52 仅存在于第一个列表中。
模块 ReLU_10_00 仅存在于第二个列表中。
模块 ReLU_2

Duplicate layer name found: ReLU_1. Renaming to ReLU_1_af
Duplicate layer name found: ReLU_1. Renaming to ReLU_1_29
Duplicate layer name found: ReLU_2. Renaming to ReLU_2_cb
Duplicate layer name found: ReLU_2. Renaming to ReLU_2_51
Duplicate layer name found: ReLU_3. Renaming to ReLU_3_1a
Duplicate layer name found: ReLU_3. Renaming to ReLU_3_c4
Duplicate layer name found: ReLU_4. Renaming to ReLU_4_bf
Duplicate layer name found: ReLU_4. Renaming to ReLU_4_a0
Duplicate layer name found: ReLU_5. Renaming to ReLU_5_0c
Duplicate layer name found: ReLU_5. Renaming to ReLU_5_bf
Duplicate layer name found: ReLU_6. Renaming to ReLU_6_0e
Duplicate layer name found: ReLU_6. Renaming to ReLU_6_a3
Duplicate layer name found: ReLU_7. Renaming to ReLU_7_68
Duplicate layer name found: ReLU_7. Renaming to ReLU_7_7a
Duplicate layer name found: ReLU_8. Renaming to ReLU_8_ff
Duplicate layer name found: ReLU_8. Renaming to ReLU_8_7b
Duplicate layer name found: ReLU_9. Renaming to ReLU_9_34
Duplicate laye

...

Duplicate layer name found: ReLU_1. Renaming to ReLU_1_dd
Duplicate layer name found: ReLU_1. Renaming to ReLU_1_fb
Duplicate layer name found: ReLU_2. Renaming to ReLU_2_0f
Duplicate layer name found: ReLU_2. Renaming to ReLU_2_dd
Duplicate layer name found: ReLU_3. Renaming to ReLU_3_62
Duplicate layer name found: ReLU_3. Renaming to ReLU_3_d7
Duplicate layer name found: ReLU_4. Renaming to ReLU_4_04
Duplicate layer name found: ReLU_4. Renaming to ReLU_4_63
Duplicate layer name found: ReLU_5. Renaming to ReLU_5_a5
Duplicate layer name found: ReLU_5. Renaming to ReLU_5_3f
Duplicate layer name found: ReLU_6. Renaming to ReLU_6_7c
Duplicate layer name found: ReLU_6. Renaming to ReLU_6_2b
Duplicate layer name found: ReLU_7. Renaming to ReLU_7_27
Duplicate layer name found: ReLU_7. Renaming to ReLU_7_34
Duplicate layer name found: ReLU_8. Renaming to ReLU_8_86
Duplicate layer name found: ReLU_8. Renaming to ReLU_8_eb
Duplicate layer name found: ReLU_9. Renaming to ReLU_9_c9
Duplicate laye

模块 ReLU_5_3f 仅存在于第二个列表中。
模块 ReLU_1_86 仅存在于第一个列表中。
模块 ReLU_13_58 仅存在于第二个列表中。
模块 ReLU_5_07 仅存在于第一个列表中。
模块 ReLU_8_eb 仅存在于第二个列表中。
模块 ReLU_12_66 仅存在于第二个列表中。
模块 ReLU_4_4e 仅存在于第一个列表中。
模块 ReLU_6_f0 仅存在于第一个列表中。
模块 ReLU_4_04 仅存在于第二个列表中。
模块 ReLU_10_2d 仅存在于第二个列表中。
模块 ReLU_15_a0 仅存在于第一个列表中。
模块 ReLU_3_96 仅存在于第一个列表中。
模块 ReLU_16_d5 仅存在于第二个列表中。
模块 ReLU_11_3e 仅存在于第一个列表中。
模块 ReLU_4_82 仅存在于第一个列表中。
模块 ReLU_16_bb 仅存在于第一个列表中。
模块 ReLU_9_67 仅存在于第一个列表中。
模块 ReLU_1_3b 仅存在于第一个列表中。
模块 ReLU_3_62 仅存在于第二个列表中。
模块 ReLU_10_80 仅存在于第一个列表中。
模块 ReLU_9_b0 仅存在于第一个列表中。
模块 ReLU_14_d7 仅存在于第二个列表中。
模块 ReLU_7_34 仅存在于第二个列表中。
模块 ReLU_10_39 仅存在于第一个列表中。
模块 ReLU_12_0e 仅存在于第一个列表中。
模块 ReLU_12_6b 仅存在于第一个列表中。
模块 ReLU_6_f8 仅存在于第一个列表中。
模块 ReLU_8_92 仅存在于第一个列表中。
模块 ReLU_8_41 仅存在于第一个列表中。
模块 ReLU_5_95 仅存在于第一个列表中。
模块 ReLU_6_7c 仅存在于第二个列表中。
模块 ReLU_15_2a 仅存在于第一个列表中。
模块 ReLU_7_be 仅存在于第一个列表中。
模块 ReLU_14_d4 仅存在于第二个列表中。
模块 ReLU_3_ca 仅存在于第一个列表中。
模块 ReLU_8_86 仅存在于第二个列表中。
模块 ReLU_1_dd 仅存在于第二个列表中。
模块 ReLU_2_dd 仅存在于第二个列表中。
模块 ReLU_16_c5 仅存在于第二个列表中。
模块 ReLU_7_

Duplicate layer name found: ReLU_1. Renaming to ReLU_1_2a
Duplicate layer name found: ReLU_1. Renaming to ReLU_1_5c
Duplicate layer name found: ReLU_2. Renaming to ReLU_2_5f
Duplicate layer name found: ReLU_2. Renaming to ReLU_2_96
Duplicate layer name found: ReLU_3. Renaming to ReLU_3_17
Duplicate layer name found: ReLU_3. Renaming to ReLU_3_36
Duplicate layer name found: ReLU_4. Renaming to ReLU_4_3c
Duplicate layer name found: ReLU_4. Renaming to ReLU_4_8b
Duplicate layer name found: ReLU_5. Renaming to ReLU_5_4b
Duplicate layer name found: ReLU_5. Renaming to ReLU_5_98
Duplicate layer name found: ReLU_6. Renaming to ReLU_6_61
Duplicate layer name found: ReLU_6. Renaming to ReLU_6_66
Duplicate layer name found: ReLU_7. Renaming to ReLU_7_dd
Duplicate layer name found: ReLU_7. Renaming to ReLU_7_a3
Duplicate layer name found: ReLU_8. Renaming to ReLU_8_f3
Duplicate layer name found: ReLU_8. Renaming to ReLU_8_55
Duplicate layer name found: ReLU_9. Renaming to ReLU_9_d0
Duplicate laye

模块 ReLU_5_09 仅存在于第一个列表中。
模块 ReLU_3_17 仅存在于第二个列表中。
模块 ReLU_15_77 仅存在于第二个列表中。
模块 ReLU_13_e0 仅存在于第一个列表中。
模块 ReLU_8_79 仅存在于第一个列表中。
模块 ReLU_9_c1 仅存在于第一个列表中。
模块 ReLU_16_8f 仅存在于第一个列表中。
模块 ReLU_13_1a 仅存在于第二个列表中。
模块 ReLU_11_b0 仅存在于第一个列表中。
模块 ReLU_9_2a 仅存在于第二个列表中。
模块 ReLU_12_fb 仅存在于第二个列表中。
模块 ReLU_10_2d 仅存在于第二个列表中。
模块 ReLU_12_ca 仅存在于第二个列表中。
模块 ReLU_16_e0 仅存在于第二个列表中。
模块 ReLU_5_98 仅存在于第二个列表中。
模块 ReLU_5_77 仅存在于第一个列表中。
模块 ReLU_6_55 仅存在于第一个列表中。
模块 ReLU_12_8b 仅存在于第一个列表中。
模块 ReLU_3_36 仅存在于第二个列表中。
模块 ReLU_16_04 仅存在于第一个列表中。
模块 ReLU_3_82 仅存在于第一个列表中。
模块 ReLU_13_9d 仅存在于第二个列表中。
模块 ReLU_15_44 仅存在于第一个列表中。
模块 ReLU_2_96 仅存在于第二个列表中。
模块 ReLU_8_1e 仅存在于第一个列表中。
模块 ReLU_8_f3 仅存在于第二个列表中。
模块 ReLU_13_0e 仅存在于第一个列表中。
模块 ReLU_14_eb 仅存在于第二个列表中。
模块 ReLU_7_09 仅存在于第一个列表中。
模块 ReLU_4_3c 仅存在于第二个列表中。
模块 ReLU_6_66 仅存在于第二个列表中。
模块 ReLU_1_5c 仅存在于第二个列表中。
模块 ReLU_15_bc 仅存在于第二个列表中。
模块 ReLU_7_dd 仅存在于第二个列表中。
模块 ReLU_6_c0 仅存在于第一个列表中。
模块 ReLU_1_2a 仅存在于第二个列表中。
模块 ReLU_10_2c 仅存在于第一个列表中。
模块 ReLU_16_5e 仅存在于第二个列表中。
模块 ReLU_14_c2 仅存在于第二个列表中。
模块 ReL

OOM when allocating 1004535808 bytes on device index=0 id='6cdff0a130be4563856b29efa63cacde'
OOM when allocating 1004535808 bytes on device index=0 id='6cdff0a130be4563856b29efa63cacde'


In [8]:
import pandas as pd
df = pd.json_normalize(result_list)

In [9]:
df["Estimate Diff"] = df["compare.est_diff"].apply(format_memory)
df

,model,batch,torch.est,torch.ground,torch.error,torch.diff,huggingface.est,huggingface.ground,huggingface.error,huggingface.diff,compare.forward,compare.backward,compare.est_diff,Estimate Diff
0,ConvNeXtTiny,10,369098752,312475648,18.12,56623104,354418688,262144000,35.20,92274688,True,True,-14680064,-14.00 MB
1,ConvNeXtTiny,50,994050048,998244352,0.42,-4194304,975175680,958398464,1.75,16777216,True,True,-18874368,-18.00 MB
2,ConvNeXtTiny,90,1614807040,1233125376,30.95,381681664,1604321280,1629487104,1.54,-25165824,True,True,-10485760,-10.00 MB
3,ConvNeXtTiny,130,2210398208,1625292800,36.00,585105408,2206203904,2281701376,3.31,-75497472,True,True,-4194304,-4.00 MB
4,ConvNeXtTiny,170,2824863744,1644167168,71.81,1180696576,2824863744,2925527040,3.44,-100663296,True,True,0,0 b
5,ConvNeXtTiny,210,3468689408,1667235840,108.05,1801453568,3468689408,3590324224,3.39,-121634816,True,True,0,0 b
6,ConvNeXtTiny,250,4106223616,4678746112,12.24,-572522496,4112515072,4280287232,3.92,-167772160,True,True,6291456,6.00 MB
7,ConvNeXtTiny,290,4720689152,1755316224,168.94,2965372928,4731174912,4945084416,4.33,-213909504,True,True,10485760,10.00 MB
8,ConvNeXtTiny,330,5368709120,3690987520,45.45,1677721600,5381292032,5599395840,3.90,-218103808,True,True,12582912,12.00 MB
9,ConvNeXtTiny,370,5911871488,5771362304,2.43,140509184,5928648704,6199181312,4.36,-270532608,True,True,16777216,16.00 MB



# Analysis of Shopshot Data between PyTorch and HuggingFace


In [10]:
model_name = model
batch = batch_size
torch_data_dir = pytorch_dir.joinpath(pytorch_dir_name_format.format(model_name, batch))
all_torch_dirs = os.listdir(torch_data_dir)
all_torch_dirs = [torch_data_dir.joinpath(d) for d in all_torch_dirs if str(d).startswith(".") is False and str(d).startswith("@") is False]
dirs_sorted = sorted(all_torch_dirs, key=lambda d: d.stat().st_ctime)

In [11]:
torch_snapshot_file = Path(filter_files(".pickle", dirs_sorted[0], fuzz=True)[0])
torch_profiler_file = Path(filter_files(".pt.trace.json", dirs_sorted[0], fuzz=True)[-1])


In [12]:
# Get HuggingFace Data
huggingface_dir_xmem = huggingface_dir.joinpath(huggingface_dir_name_format_xmem.format(model_name, batch))
huggingface_dir_cuda = huggingface_dir.joinpath(huggingface_dir_name_format_cuda.format(model_name, batch))
huggingface_dir_llm = huggingface_dir.joinpath(huggingface_dir_name_format_llm.format(model_name, batch))
huggingface_profiler_file_xmem = filter_files(".pt.trace.json", huggingface_dir_xmem, fuzz=True)[-1]
huggingface_profiler_file_llm = filter_files(".pt.trace.json", huggingface_dir_llm, fuzz=True)[-1]
huggingface_snapshot_file_xmem = Path(filter_files(".pickle", huggingface_dir_cuda, fuzz=True)[-1])

torch_snapshot_file

PosixPath('/Users/jiaboshi/Documents/101-Data/002-xMem-LLM/001-PyTorch/recurrence-VGG16-SGD-170-1/20241126-015141-d180/results/snapshot/snapshot_result-1732585963.pickle')

In [13]:
torch_snapshot = SnapshotAnalyser(torch_snapshot_file)
hugging_snapshot = SnapshotAnalyser(huggingface_snapshot_file_xmem)

In [14]:
torch_profiler = ProfilerDataProcessing(torch_profiler_file)

In [15]:
huggingface_profiler = ProfilerDataProcessing(huggingface_profiler_file_llm)
huggingface_profiler_file_llm

'/Users/jiaboshi/Documents/101-Data/002-xMem-LLM/002-HuggingFace/VGG16-170-LLM/results/callback/Profiler/Glaswigian-Researcher_123964.1739486916203614949.pt.trace.json'

In [16]:
layer_in_torch = torch_profiler.get_iteration(2).layer_summary()
layer_in_hugging = huggingface_profiler.get_iteration(2).layer_summary()

In [17]:
_, torch_estimator = get_xmem_memory(torch_profiler_file, batch_size, max_gpu_memory)
_, hugging_estimator = get_xmem_memory(huggingface_profiler_file_llm, batch_size, max_gpu_memory, huggingface_enabled=True)

In [18]:
training_memorys = torch_estimator.training_memory(iteration_index=2)
model_memory = torch_estimator.model_memory(iteration_index=2)
data_memory = torch_estimator.data_memory(iteration_index=2)
estimated_instance, estimated_result = torch_estimator.estimate_memory_blocks(training_memorys)
estimated_model_instance, estimated_model_result = torch_estimator.estimate_memory_blocks(model_memory)
estimated_data_instance, estimated_data_result = torch_estimator.estimate_memory_blocks(data_memory)

print(f"Model Memory: {format_memory(estimated_model_result['memory']['segment'])}\n"
      f"Training Memory: {format_memory(estimated_result['memory']['segment'])}\n"
      f"Data Memory: {format_memory(estimated_data_result['memory']['segment'])}")


Model Memory: 534.00 MB
Training Memory: 3.28 GB
Data Memory: 18.00 MB


In [19]:
hugging_memory = hugging_estimator.training_memory(iteration_index=2, zero_grad=False)
hugging_model_memory = hugging_estimator.model_memory(iteration_index=2)
hugging_data_memory = hugging_estimator.data_memory(iteration_index=2)
hugging_estimated_instance, hugging_estiamted_result = hugging_estimator.estimate_memory_blocks(hugging_memory)
hugging_estimated_memory_instance, hugging_model_memory_result = hugging_estimator.estimate_memory_blocks(hugging_model_memory)
hugging_estimatod_data_instance, hugging_data_memory_result = hugging_estimator.estimate_memory_blocks(hugging_data_memory)

print(f"Model Memory: {format_memory(hugging_model_memory_result['memory']['segment'])}\n"
      f"Training Memory: {format_memory(hugging_estiamted_result['memory']['segment'])}\n"
      f"HuggingFace Data Memory: {format_memory(hugging_data_memory_result['memory']['segment'])}")

Model Memory: 534.00 MB
Training Memory: 3.28 GB
HuggingFace Data Memory: 18.00 MB
